In [ ]:
import time
import threading
import numpy as np
import cv2
import matplotlib.pyplot as plt
from robomaster.robot import Robot
from skimage.color import rgb2hsv

# ==========================================
# --- COSTANTI E PARAMETRI ---
# ==========================================
KP_YAW             = 1.5
MAX_Z              = 30.0
OBSTACLE_DIST_MM   = 200
BLUE_STOP_MM       = 15
DODGE_Y_SPEED      = 0.45
FORWARD_SPEED      = 0.3
DODGE_LATERAL_TIME = 0.7    # FIX: era 1.2s, troppo lungo
MAX_FWD_TIME       = 7.0
MIN_FWD_TIME       = 1.0
RETURN_FACTOR      = 1.20
MIN_PIXELS         = 10000

# ==========================================
# --- STATO GLOBALE THREAD-SAFE ---
# ==========================================
_lock               = threading.Lock()
accumulated_yaw     = 0.0
last_yaw            = None
current_distance    = 2000.0
readings            = []
last_turn_was_right = False   # FIX: ora usato per TUTTI gli ostacoli

# ==========================================
# --- CALLBACKS SENSORI ---
# ==========================================
def attitude_handler(attitude_info):
    global accumulated_yaw, last_yaw
    yaw = attitude_info[0]
    if last_yaw is None:
        last_yaw = yaw
        return
    delta = yaw - last_yaw
    if delta > 180:    delta -= 360
    elif delta < -180: delta += 360
    with _lock:
        accumulated_yaw += delta
    last_yaw = yaw

def distance_handler(data):
    global current_distance, readings
    dist = data[0]
    with _lock:
        current_distance = dist if dist != 65534 else 2000.0
        readings.append((abs(accumulated_yaw), current_distance))

# ==========================================
# --- HELPER YAW ---
# ==========================================
def get_z_correction(max_z: float = MAX_Z) -> float:
    with _lock:
        yaw = accumulated_yaw
    return float(np.clip(-KP_YAW * yaw, -max_z, max_z))

def reset_yaw_reference():
    global accumulated_yaw, last_yaw
    with _lock:
        accumulated_yaw = 0.0
        last_yaw = None

# ==========================================
# --- VISIONE ---
# ==========================================
def detect_color(img_rgb: np.ndarray):
    hsv = rgb2hsv(img_rgb.astype(np.float32) / 255.0)
    H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    counts = {
        "blue":   np.sum((H >= 0.55) & (H <= 0.70) & (S > 0.40) & (V > 0.20)),
        "black":  np.sum(V < 0.20),
        "orange": np.sum((H >= 0.02) & (H <= 0.15) & (S > 0.50) & (V > 0.20)),
    }
    dominant = max(counts, key=counts.get)
    return dominant if counts[dominant] > MIN_PIXELS else None

def guess_unknown_color(img_rgb: np.ndarray) -> str:
    h, w = img_rgb.shape[:2]
    roi = img_rgb[h // 4 : 3 * h // 4, w // 4 : 3 * w // 4]
    R, G, B = np.mean(roi, axis=(0, 1))
    if R > G and R > B:                  return "Rosso/Marrone"
    if G > R and G > B:                  return "Verde"
    if B > R and B > G:                  return "Azzurro/Viola"
    if R > 200 and G > 200 and B > 200:  return "Bianco"
    return "Sconosciuto"

# ==========================================
# --- HELPER: Movimento con correzione Yaw ---
# ==========================================
def drive_corrected(robot, x: float, y: float, duration: float,
                    label: str = "", early_exit_fn=None):
    start = time.time()
    while time.time() - start < duration:
        robot.chassis.drive_speed(x=x, y=y, z=get_z_correction())
        frame = robot.camera.read_video_frame(strategy="newest")
        if frame is not None:
            disp = frame.copy()
            if label:
                cv2.putText(disp, label, (40, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.imshow("RoboMaster - Live", disp)
            cv2.waitKey(1)
            if early_exit_fn is not None and early_exit_fn(frame[:, :, ::-1]):
                break
        time.sleep(0.05)
    robot.chassis.drive_speed(x=0, y=0, z=0)

# ==========================================
# --- DODGE ---
# ==========================================
def dodge_obstacle(robot, direction_y: float, obstacle_color: str):
    side = "DESTRA" if direction_y > 0 else "SINISTRA"
    print(f"\n[DODGE] ▶ {side} | ostacolo: {obstacle_color}")

    # -- FASE 1: Stop + schivata laterale --
    robot.chassis.drive_speed(x=0, y=0, z=0)
    time.sleep(0.12)
    drive_corrected(robot, x=0.0, y=direction_y,
                    duration=DODGE_LATERAL_TIME, label=f"SCHIVATA {side}")
    reset_yaw_reference()
    time.sleep(0.1)

    # -- FASE 2: Gimbal verso l'ostacolo --
    gimbal_yaw = 65 if direction_y > 0 else -65
    try:
        robot.gimbal.moveto(yaw=gimbal_yaw, pitch=0).wait_for_completed()
    except Exception as e:
        print(f"[WARN] Gimbal: {e}")

    # -- FASE 3: Avanzamento con conferma visiva --
    print(f"[DODGE] ▶ Avanzamento, cerco {obstacle_color}...")
    obstacle_was_seen = False
    start_fwd = time.time()

    while time.time() - start_fwd < MAX_FWD_TIME:
        robot.chassis.drive_speed(x=FORWARD_SPEED, y=0.0, z=get_z_correction())
        frame = robot.camera.read_video_frame(strategy="newest")

        if frame is not None:
            rgb     = frame[:, :, ::-1]
            seen    = detect_color(rgb)
            elapsed = time.time() - start_fwd

            disp = frame.copy()
            cv2.putText(disp, f"SUPERO {obstacle_color} | {elapsed:.1f}s",
                        (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(disp, f"Vedo: {seen} | Confermato: {obstacle_was_seen}",
                        (30, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)
            cv2.imshow("RoboMaster - Live", disp)
            cv2.waitKey(1)

            if elapsed >= MIN_FWD_TIME:
                if obstacle_color == "Sconosciuto":
                    if elapsed > 2.8:
                        print("[DODGE] ✓ Timeout sconosciuto → rientro.")
                        break
                else:
                    if seen == obstacle_color:
                        obstacle_was_seen = True
                    elif obstacle_was_seen:
                        print(f"[DODGE] ✓ {obstacle_color} sparito → rientro.")
                        break
        time.sleep(0.05)

    robot.chassis.drive_speed(x=0, y=0, z=0)
    time.sleep(0.15)

    # -- FASE 4: Gimbal dritto + rientro --
    try:
        robot.gimbal.recenter().wait_for_completed()
    except Exception as e:
        print(f"[WARN] Gimbal recenter: {e}")

    reset_yaw_reference()
    time.sleep(0.1)
    drive_corrected(robot, x=0.0, y=-direction_y * RETURN_FACTOR,
                    duration=DODGE_LATERAL_TIME, label="RIENTRO AL CENTRO")
    reset_yaw_reference()
    time.sleep(0.15)
    print("[DODGE] ✓ Manovra completata.\n")

# ==========================================
# --- OVERLAY VIDEO ---
# ==========================================
def draw_overlay(frame, color_seen, state_msg):
    display = frame.copy()
    with _lock:
        dist = current_distance
        yaw  = accumulated_yaw
    lines = [
        (f"Stato: {state_msg}",      (0, 255, 0)),
        (f"Colore: {color_seen}",    (255, 255, 0)),
        (f"Dist: {dist:.0f} mm",     (0, 255, 255)),
        (f"Yaw: {yaw:+.1f} deg",     (255, 100, 100)),
    ]
    for i, (text, color) in enumerate(lines):
        cv2.putText(display, text, (10, 30 + i * 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    bar_len   = int(np.clip(dist / OBSTACLE_DIST_MM * 200, 0, 200))
    bar_color = (0, 255, 0) if dist > OBSTACLE_DIST_MM else (0, 0, 255)
    cv2.rectangle(display, (10, 155), (10 + bar_len, 175), bar_color, -1)
    cv2.rectangle(display, (10, 155), (210, 175), (200, 200, 200), 1)
    return display

# ==========================================
# --- MAIN ---
# ==========================================
robot = Robot()
robot.initialize(conn_type="sta", sn="LOCAL")

try:
    print("Inizializzazione sensori e camera...")
    robot.chassis.sub_attitude(freq=50, callback=attitude_handler)
    robot.sensor.sub_distance(freq=10, callback=distance_handler)
    robot.camera.start_video_stream(display=False, resolution="720p")
    try:
        robot.gimbal.recenter().wait_for_completed()
    except:
        pass
    time.sleep(1.5)
    reset_yaw_reference()
    print("\nRobot pronto! Premi Q per fermare.\n")

    while True:
        frame = robot.camera.read_video_frame(strategy="newest")
        if frame is None:
            time.sleep(0.01)
            continue

        rgb_frame  = frame[:, :, ::-1]
        color_seen = detect_color(rgb_frame)

        with _lock:
            dist = current_distance

        # --- 1. TARGET BLU ---
        if color_seen == "blue":
            if dist > BLUE_STOP_MM:
                state_msg = "Vedo BLU: avvicinamento..."
                speed_x   = 0.20 if dist > 100 else 0.05
                robot.chassis.drive_speed(x=speed_x, y=0.0,
                                          z=get_z_correction(max_z=15.0))
            else:
                state_msg = "TARGET BLU RAGGIUNTO!"
                robot.chassis.drive_speed(x=0, y=0, z=0)

        # --- 2. DODGE ---
        elif dist <= OBSTACLE_DIST_MM:
            robot.chassis.drive_speed(x=0, y=0, z=0)

            # FIX: direzione scelta per ALTERNANZA su tutti gli ostacoli
            direction_y = +DODGE_Y_SPEED if not last_turn_was_right else -DODGE_Y_SPEED

            if color_seen == "black":
                state_msg = f"NERO → {'DESTRA' if direction_y > 0 else 'SINISTRA'}"
                dodge_obstacle(robot, direction_y=direction_y, obstacle_color="black")

            elif color_seen == "orange":
                state_msg = f"ARANCIONE → {'DESTRA' if direction_y > 0 else 'SINISTRA'}"
                dodge_obstacle(robot, direction_y=direction_y, obstacle_color="orange")

            else:
                guessed   = guess_unknown_color(rgb_frame)
                state_msg = f"Ostacolo ({guessed}) → {'DESTRA' if direction_y > 0 else 'SINISTRA'}"
                dodge_obstacle(robot, direction_y=direction_y, obstacle_color="Sconosciuto")

            last_turn_was_right = (direction_y > 0)   # aggiorna DOPO il dodge

        # --- 3. AVANZAMENTO LIBERO ---
        else:
            state_msg = f"Avanzamento | vedo: {color_seen}" if color_seen else "Avanzamento"
            robot.chassis.drive_speed(x=FORWARD_SPEED, y=0.0, z=get_z_correction())

        cv2.imshow("RoboMaster - Live", draw_overlay(frame, str(color_seen), state_msg))
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\nChiusura richiesta.")
            break
        time.sleep(0.05)

except KeyboardInterrupt:
    print("\nInterruzione manuale.")
finally:
    print("\nSpegnimento...")
    robot.chassis.drive_speed(x=0, y=0, z=0)
    robot.sensor.unsub_distance()
    robot.chassis.unsub_attitude()
    cv2.destroyAllWindows()
    try:
        robot.camera.stop_video_stream()
        time.sleep(0.5)
        robot.close()
    except:
        pass

# ==========================================
# --- GRAFICO FINALE ---
# ==========================================
print("\nGenerazione grafico...")
if readings:
    dists_c = [d if d < 1999 else np.nan for _, d in readings]
    plt.figure(figsize=(12, 5))
    plt.plot(range(len(dists_c)), dists_c, '-o', markersize=3,
             color="steelblue", label="Distanza ToF")
    plt.axhline(y=OBSTACLE_DIST_MM, color='red',   linestyle='--',
                label=f"Soglia dodge ({OBSTACLE_DIST_MM} mm)")
    plt.axhline(y=BLUE_STOP_MM,     color='green', linestyle='--',
                label=f"Stop blu ({BLUE_STOP_MM} mm)")
    plt.title("Profilo distanza durante il percorso")
    plt.xlabel("Campioni ToF")
    plt.ylabel("Distanza (mm)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Nessun dato acquisito.")

Inizializzazione sensori e camera...

Robot pronto! Premi Q per fermare.


[DODGE] ▶ DESTRA | ostacolo: black
[DODGE] ▶ Avanzamento, cerco black...
[DODGE] ✓ Manovra completata.



Exception in thread Thread-3 (_task):
Traceback (most recent call last):
  File "c:\Users\manuc\miniconda3\envs\aidrones\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\manuc\miniconda3\envs\aidrones\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\manuc\miniconda3\envs\aidrones\Lib\site-packages\robomaster\conn.py", line 300, in _task
    msg = super().recv()
  File "c:\Users\manuc\miniconda3\envs\aidrones\Lib\site-packages\robomaster\conn.py", line 183, in recv
    data, host = self._sock.recvfrom(2048)
                 ~~~~~~~~~~~~~~~~~~~^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host



Interruzione manuale.

Spegnimento...


2026-03-20 17:28:46,212 ERROR client.py:163 Client: send_sync_msg wait msg receiver:0900, cmdset:0x48, cmdid:0x04 timeout!
2026-03-20 17:28:49,217 ERROR client.py:163 Client: send_sync_msg wait msg receiver:0900, cmdset:0x48, cmdid:0x04 timeout!
2026-03-20 17:28:52,238 ERROR client.py:163 Client: send_sync_msg wait msg receiver:0100, cmdset:0x3f, cmdid:0xd2 timeout!
